# Downloading and parsing EDGAR 10-k filings


This file serves to download the 10-K filings of the U.S. S&P 500 firms from the [EDGAR website](https://www.sec.gov/). How to read a 10-K filing can be taken from this [website](https://www.investor.gov/introduction-investing/general-resources/news-alerts/alerts-bulletins/investor-bulletins/how-read).                   


<div class="alert-warning">
Libraries
</div>
First, we Import all necessary libraries. These inlcude "os" and "pathlib" to set and handle working directories, "edgar" and "refinitiv.data" to access the needed data via the API servers, "pandas" and "numpy" for data handling and calculations, "pickle" to save the prepared data as memory efficient pickle files,"re" and "lxml" to handle HTML-files and text/string data, and "spacy" for NLP tasks. 

In [ ]:
import os
from pathlib import Path
from edgar import *
import pandas as pd
import numpy as np
import lseg.data as rd
import re
import lxml.html
import pickle

<div class="alert-warning">
Set the working directory
</div>

In [ ]:
os.chdir('../../data')

<div class="alert-info">
Step 1: Retrieve the CIK codes for all S&P 500 firms from Refinitiv
</div>

First, we open a new Refinitiv session (make sure that the desktop APP is open)

In [ ]:
rd.open_session()

Next, we request the necessary S&P 500 data from the Refinitiv Data API, including CIK codes and company names. This is done for each year separatly including the years 2012 to 2023. Each year the companies listed on the 31st december are considered.

In [ ]:
CIK_df = pd.DataFrame()

for i in range(2012,2024):
    universe = f"0#.SPX({i}-12-31)"
    if CIK_df.empty == True:
        CIK_df = rd.get_data(universe, ["TR.CIKNUMBER","TR.CompanyName", "TR.CUSIP"])
        CIK_df["Index_year"] = i
    else:
        CIK_tmp = rd.get_data(universe, ["TR.CIKNUMBER","TR.CompanyName", "TR.CUSIP"])
        CIK_tmp["Index_year"] = i
        CIK_df = pd.concat([CIK_df, CIK_tmp])

# Save the dataframe as a pickle file
CIK_df.to_pickle("SP_500_CIK.pkl")

# In addition, we save a version that just includes the CIK, CUSIP, and RIC codes, and the company name. 
# Duplicate entries are removed, so that each CIK, CUSIP, and RIC code is only listed once.
CIK_df_final = CIK_df[['Instrument', 'CIK Number', 'CUSIP', 'Company Name']]
CIK_df_final = CIK_df_final.drop_duplicates(subset=['CIK Number'])
CIK_df_final.to_pickle("SP_500_CIK_final.pkl")

Note: There are more than 500 CIK numbers per year. The reason is that some companies have multiple CIK codes. This is, however, no problem, as the 10-K Filings are just submitted for one these CIK codes per company. The rest will be automatically ignored when filtering the filings by S&P 500 CIK codes at a later stage.

Lastly, we can close the Refinitiv session.

In [ ]:
rd.close_session()

<div class="alert-info">
Step 2: Access the 10-K filings from the EDGAR API
</div>

First, we need to tell the SEC who we are.

In [ ]:
# Set the identity for the Edgar API. This is required to access the SEC's EDGAR database. Please replace the name and email address with your own.
# For example, you can use your name and email address as follows: "Firstname Lastname Emailaddress"

set_identity("Firstname Lastname email@example.com")

Next, we define a function that creates a filings dataframe for each year and then loops through all accession numbers of S&P 500 firms contained in the dataframe and downloads the primary document (10-k) for each filing.

In [ ]:
def get_files(start_year :int , end_year :int ,
              cik_codes: pd.DataFrame, odirect:str,
              html = True):
    """
    Downloads SEC filings for specific companies: \n
    start_year -> First Year to download \n
    end_year -> Last Year to download \n
    cik_codes -> List of CIK codes for S&P 500 firms for which filings should be downloaded \n
    odirect -> Directory the filings will be downloaded to \n
    html -> If True, the filings will be downloaded as HTML files. If False, the filings will be downloaded as markdown files. \n
    """
    print('Downloading Filings')
    #Set the filter to receive the filings information (form, cik, accession number)
    #for the years start_year to end_year. We also do not include amendents.
    filings = get_filings(year=range(start_year, end_year+1), form="10-K",amendments=False)

    #If the html option is set to False, the filings will be downloaded in another directory
    if html == False:
        odirect = odirect + "/markdown"        

    #Transform the filings summary object to a pandas dataframe
    filings_df = filings.to_pandas()

    #Adjust the length of the CIK codes to match 10 entries. This helps to find the S&P 500 firms.
    filings_df["cik"] = filings_df["cik"].astype(str).str.zfill(10)

    cik_codes["CIK Number"] = cik_codes["CIK Number"].astype(str).str.zfill(10)

    #Add filing year to the dataframe
    filings_df["filing_year"] = filings_df["filing_date"].astype(str).str[:4].astype(int)

    #Create unique cik_year keys to loop through each key-value.
    filings_df["filing_key"] = filings_df["cik"].astype(str) + "_" + filings_df["filing_year"].astype(str)

    # S&P 500 filings information will be saved in a separate dataframe. This dataframe will be saved as a pickle file for later use.
    S_and_P_500_filings = filings_df[filings_df["cik"].isin(cik_codes["CIK Number"])].reset_index()

    S_and_P_500_filings = S_and_P_500_filings[["company","cik", "form", "filing_year", "accession_number","filing_key"]]
            
    S_and_P_500_filings.to_pickle("S_and_P_500_filings_info.pkl")
    
    #Loop through each filing year and save the 10-K file as HTML or Markdown text
    for i in range(start_year,end_year+1):
        #First, filter the filing dataframe for the respective filing year
        filings_tmp = filings_df[filings_df["filing_year"] == i].reset_index()
        #Second, filter the S&P 500 CIK codes for the preceeding year end date (31.12.2020 for filing year 2021) 
        cik_tmp = cik_codes[cik_codes["Index_year"] == i-1].reset_index()
        #Third, filter filings for matching CIK codes with the S&P 500 index
        S_and_P_500_filings = filings_tmp[filings_tmp["cik"].isin(cik_tmp["CIK Number"])].reset_index()

        S_and_P_500_filings = S_and_P_500_filings[["company","cik", "form", "filing_year", "accession_number","filing_key"]]

        for i in range(0, len(S_and_P_500_filings["filing_key"])):
            #save the information for the considered firm as a temporary variable
            firm_info = S_and_P_500_filings.iloc[i]

            #check whether the directory exists for the filing year
            # and create one if it does not
            download_path = os.path.join(odirect,str(firm_info["filing_year"]))
            if not os.path.exists(download_path):
                os.makedirs(download_path)

            #Access the 10-K filing via its acccession number
            filing = get_by_accession_number(firm_info['accession_number'])

            #Retrieve reporting date of the filing (save it as year, month, and day). If the filing date is not available, set it to "NA"
            try: 
                 reporting_year = filing.filing_date.year
                 reporting_month = filing.filing_date.month 
                 reporting_day = filing.filing_date.day
            except AttributeError:
                 reporting_year = "NA"
                 reporting_month = "NA"
                 reporting_day = "NA"

            # Create the filename to download the file to. Reporting month shold be 2 digits long, so we add a leading zero if necessary. 
            # The file name is constructed as follows: CIK_ReportingMonth_ReportingYear.txt
            file_name = str(firm_info["cik"])+"_"+str(reporting_month).zfill(2)+"_"+str(reporting_year)+".txt"
            output_file_path = os.path.join(download_path,file_name)

            #Add "iXBRL" to the file name if the document is filed as iXBRL 
            #and without if it is filed as HTML
            try: 
                if filing.primary_documents[0].document.endswith("iXBRL"):
                    output_file_path = output_file_path.replace('.txt', '_iXBRL.txt')

            except AttributeError:
                pass

            # Check to make sure the file hasn't already been downloaded. If not, download it.
            if not (os.path.isfile(output_file_path) and os.access(output_file_path, os.R_OK)):
                # Print the name of the file to be downloaded
                print(f"Downloading: {firm_info['form']}-File for company '{firm_info['company']}' \n\
                with filing year {firm_info['filing_year']} \n\
                and accession_number: {firm_info['accession_number']}")                

                # Save the html/markup text as text file
                if html == True:
                    filing_text = filing.html()
                    #Open the file in write mode and write the HTML content into it
                    with open(output_file_path, 'w', encoding='utf-8') as file:
                        file.write(filing_text)
                else:
                    try:
                        filing_text = filing.markdown()
                    except AttributeError:
                        filing_text = ""
                    #save the markdown content as .txt file without encoding
                    with open(output_file_path, 'w', encoding='utf-8') as file:
                        file.write(filing_text)                                 
                
            else:
                # Print that file has already been downloaded
                print(f"File has already been downloaded: {firm_info['form']}-File for company '{firm_info['company']}' \
                with filing year {firm_info['filing_year']} \
                and accession_number: {firm_info['accession_number']}",end='\n')  
    return

Now, we can define the download path, load the S&P 500 CIK codes, and download the filings from 2013 to 2023.

In [ ]:
download_folder = "10K"

with open('SP_500_CIK.pkl','rb') as path_name:
    CIK_df = pickle.load(path_name)

get_files(start_year=2013, end_year=2023, cik_codes=CIK_df,
          odirect=download_folder, html=True)

<div class="alert-info">
Step 2: Parsing of 10-K filings
</div>

Although the extraction was sucessful, the output is a HTML/iXBRL source code. Now, we want to convert this HTML/iXBRL code to plain text.

<div class="alert-info">
Step 2.1: Define function to extract the position of the ITEM headings
</div>

Before extracing the raw text from the HTML source code, we define a function that is able to find the positon of the section titles/headings. This function can be later used to extract information from specific parts of the annual report. It is coded in such a way that it can identify most section headings. There might be a few special cases, in which this code does not work (in our case this is holds for about 80 reports out of 5338).

In [ ]:
# Define the regular expression to identify the position of Items
regex_items = re.compile(r'((Item|ITEM|Items|ITEMS)(\s|&#160;|&nbsp;)+(\d{1,2}\w{0,1})\.{0,1}(\s|&#160;|&nbsp;)*(\:*\[*\s*\w*(&#x*\d+;)*\’*\'*\-*\,*\.*\]*)*)|((Part|PART)(\s|&#160;|&nbsp;)*(\d{0,1}\w{0,3}))')
# Define regular expression patterns that identify HTML text styles commonly used to display section headings
html_styles = [
                # b tag; bold text
                r"<b>(?P<value>.+?)<\/b>",
                # u tag; underlined text
                r"<u>(?P<value>.+?)<\/u>",
                # strong tag; important text    
                r"<strong[^>]*>(?P<value>.+?)<\/strong>",
                # center tag; centered text
                r"<center[^>]*>(?P<value>.+?)<\/center>",
                # any tag that has an attribute (" style ") with 'font - weight : bold or 700 ' value
                r"<(?P<tag>[\w-]+)\b[^>]*(FONT-WEIGHT|font-weight):\s*(bold|700)[^>]*>(?P<value>.*?)<\/(?P=tag)>",
                # any tag that has an attribute (" style ") with 'text - decoration : underline ' value
                r"<(?P<tag>[\w-]+)\b[^>]*(TEXT-DECORATION|text-decoration):\s*(UNDERLINE|underline)[^>]*>(?P<value>.*?)<\/(?P=tag)>",
                # any tag that has an attribute (" style ") with 'font - size : > 11-19 ' value
                r"<(?P<tag>[\w-]+)\b[^>]*(FONT-SIZE|font-size):\s*1[1-9](PT|pt){0,1}[^>]*>(?P<value>.*?)<\/(?P=tag)>",
                # em tag; emphasized text
                r"<em>(?P<value>.+?)<\/em>"]

# Define the function that for the given regex HTML style patterns and HTML source (document) returns all the (text) values of
# HTML elements that match that HTML style along with their positions (indexes) in the document's HTML source code

def get_section_position(html_source:str):
    # Use finditer to find matches to the regex
    matches = regex_items.finditer(html_source)
    
    # Save the matches as dictionary and then transform it into a data frame
    results_items = [{'text':m.group(),'start_position':m.start()} for 
                     m in matches]
    results_items = pd.DataFrame(results_items)

    #create an empty dataframe to save all possible section headings
    results_header_df = pd.DataFrame()
    
    # Loop through the possible regex of section headers and find the matches 
    for html_style in html_styles:
        # creates a regular expression from the input HTML style pattern
        html_style_regex = re.compile(html_style , re.IGNORECASE| re.DOTALL)

        # Use finditer to find matches
        style_matches = html_style_regex.finditer(html_source)

        # Save the matches as dictionary and then transform it into a data frame
        results_header = [{'text':m['value'],'start_position':m.start()} for 
                          m in style_matches]
        
        results_header = pd.DataFrame(results_header)

        if results_header_df.empty == True:
             results_header_df = results_header
        else:
             results_header_df = pd.concat([results_header_df, results_header], ignore_index=True)
    
    #Remove any HTML Tags in the header dataframe (e.g. 0001047469-14-001154)
    try:
        results_header_df["text"] = results_header_df["text"].replace('(<br\/>\s*|<BR\/>\s*|<br>\s*|<BR>\s*)','',regex=True)
        results_header_df["text"] = results_header_df["text"].replace('(<a.*?>(.*?|\n)<\/a>|<A.*?>(.*?|\n)<\/A>)','',regex=True)
        results_header_df["text"] = results_header_df["text"].replace('(<font.*?>|<\/font>|<FONT.*?>|<\/FONT>)','',regex=True)
        results_header_df["text"] = results_header_df["text"].replace('(<span.*?>|<\/span>|<SPAN.*?>|<\/SPAN>)','',regex=True)
        results_header_df["text"] = results_header_df["text"].replace('(<div.*?>|<\/div>|<DIV.*?>|<\/DIV>)','',regex=True)
    except KeyError:
        pass

    # Get rid of unnesesary characters/uni codes from the dataframes
    results_header_df = results_header_df.replace('&#x*\d+;',' ',regex=True)
    results_items = results_items.replace('&#x*\d+;',' ',regex=True)
    results_header_df = results_header_df.replace('&nbsp;',' ',regex=True)
    results_items = results_items.replace('&nbsp;',' ',regex=True)
    results_header_df = results_header_df.replace('^\s+|\s+$','',regex=True)
    results_items = results_items.replace('^\s+|\s+$','',regex=True)
    results_header_df = results_header_df.replace('\s+',' ',regex=True)
    results_items = results_items.replace('\s+',' ',regex=True)
    results_header_df = results_header_df.replace('\.','',regex=True)
    results_items = results_items.replace('\.','',regex=True)
    results_header_df = results_header_df.replace('\>','',regex=True)
    results_items = results_items.replace('\>','',regex=True)
    results_header_df = results_header_df.replace('\:','',regex=True)
    results_items = results_items.replace('\:','',regex=True)

    #Filter the dataframe of section headers that match with the items found before
    try: 
         results = results_header_df[results_header_df["text"].isin(results_items["text"])].copy()
    except KeyError:
         results = pd.DataFrame()
         return results 

    # Split up the text into Item + Number and Text, in order to delet the first occurence
    # This is most likely part of the Table of Content
    results["item_number"] = results["text"].str.upper().str.slice(0,7)
    results["item_text"] = results["text"].str.slice(7)

    #we will use the Pandas dataframe .drop_duplicates() method to only keep the last Item 
    #matches in the dataframe and drop the rest
    results["item_number"] = results["item_number"].replace('\s+','',regex=True)
    results = results.sort_values('start_position', ascending=True).drop_duplicates(subset=['item_number'], keep="last").reset_index(drop=True)

    # Add end-position for each section 
    results["end_position"] = results['start_position'].shift(-1)
    results.at[len(results) - 1, 'end_position'] = len(html_source)
    results["end_position"] = results["end_position"].astype(int)

    return results

<div class="alert-info">
Step 2.2: Extracting the raw text from the HTML / iXBRL files
</div>

In a first step we define a loop that extracts the text content of the submission files, which are provided as a html source code. This source code typically includes a header with format-related information, e.g. "</!-- Document created using WebFilings 1 -->" or the encoding (UTF-8, ASCII), and the main document "body" including: a title page with a predefined structure, a table of content, and multiple pages of text and tables of financial figures (typically divided into 16 items). We want to extract just the text information from item 1-16 (starting with Item/Part 1), excluding the title page and financial tables (as we are just interested in the additional information of the text content). To do so, we can use the function from step 2.1 and start exctracting the text from Part 1 / Item 1 to end. In addition, we will remove special character (e.g. '\"#*+•/:;<=>?@[\]^_—`{|}~™†®☒☐■☑●◦”“'), remaining unicodes, any white space (tabs & newlines), and any empty space (i.e. if two or more spaces appear next to each other they will be replaced by one single space). 

__Note__: This is just a preliminary pre-processing step. At a later stage, depending on the applied NLP-Method, we need further pre-processing steps.


First, we will define a function that helps to encode the HTML files.

In [ ]:
def get_text_from_html (html:str):
    # creates an lxml document object
    doc = lxml.html.fromstring(html)
    # optional : removes header from the HTML source code
    for header in doc.xpath('.//header'):
        header.getparent().remove(header)
    # optional : removes tables from the HTML source code
    # E.g. Number/Total characters > 10% (See Loughran p.40)?
    #table.getparent().remove(table)
    tables_obj = doc.xpath('.//table')
    for table in range(1,len(tables_obj)):
        text = []
        for element in tables_obj[table].xpath('.//tr/td//text()'):
            text.append(f"{element}")
        text = ' '.join(text).replace("\n", " ").replace("\t", " ").replace('\xa0',' ').strip()
        count_number = len(re.findall(r'\b\d+(?:[.,]\d{3})*(?:[.,]\d+)?\b',text))
        count_text = len(text.split())
        if (count_text == 0 or count_number/count_text >= 0.1 ):
             tables_obj[table].getparent().remove(tables_obj[table])

    # optional : Page numbering and table of content ref often with tag <a>  
    for a_tag in doc.xpath('.//a'):
        a_tag.getparent().remove(a_tag)
    # preserves line breaks
    # HTML tags in the list below should be followed by
    # new line character
    for tag in [".//p", ".//div", ".//span", ".//br", ".//h1", ".//h2", ".//h3", 
                ".//h4", ".//h5", ".//font"]:
        # finds all elements for a given tag
        for element in doc.findall(tag):
            # if the text value is non - empty 
            #and length is smaller than 3 (page numbering)
            #remove the string. Else add a
            # new line character ( line break )            
            if element.text:
                if element.text.strip().isnumeric() == True:
                     element.text = ""
                else:
                    element.text = element.text + "\n"
            # else creates a text value with a
            # new line character
            else:
                element.text = ""             
    # extracts and output text from the HTML source code
    return doc.text_content()


Next, we will loop through the filings and extract the text.

The frist code is used for the BoW approach.

In [ ]:
# Create new folder to save intermediate files
if not os.path.exists("BoW/Intermediate_datasets"):
    os.makedirs("BoW/Intermediate_datasets")

# Create a new dataframe to save the pre-processed files
corpus_df = pd.DataFrame()

#Step option to further clean up the text
further_clean_up = True

#Set option whether to save a single cleaned 10-k filing as a .txt file
save_single_file = False

#Loop through each year, pre-process the filings, and save them into the dataframe
for i in range(2013,2024):
    #Define the input-path (i.e. where you saved the 10-K filings to in Step 1)
    os_path = "10K/" + str(i)
    #Define the output-path
    output_path = "10K/" + str(i) + "/cleaned/"
    #Extract a list of all files included in the respective year-folder
    html_docs = os.listdir(os_path)
    if 'cleaned' in html_docs:
        html_docs.remove('cleaned')

    #Create a temporary dataframe for the respective filing-year
    corpus_tmp_df = pd.DataFrame()

    #Define first variables like "file_name", "filing_year", "CIK codes", and a empty column for the pre-processed filing text
    corpus_tmp_df["file_name"] = html_docs
    corpus_tmp_df["filing_year"] = i
    corpus_tmp_df["CIK"] = corpus_tmp_df["file_name"].str.slice(0,10)
    corpus_tmp_df["reporting_month"] = corpus_tmp_df["file_name"].str.slice(11,13)
    corpus_tmp_df["reporting_year"] = corpus_tmp_df["file_name"].str.slice(14,18)
    corpus_tmp_df["filing_text"] = None
    corpus_tmp_df["section_error"] = 0

    #loop through each filing for the respective filing year and pre-proccess it.
    for file_name in html_docs:

        file_path = os.path.join(os_path,file_name)
        with open(file_path, 'r', encoding='utf-8') as txt_file:
            html_text = txt_file.read()

        #get the position of the single items
        item_position = get_section_position(html_text)
        start_index = pd.NA

        #index for start items
        if item_position.empty:
            corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name, "section_error"] = 1
        elif (item_position["item_number"] == ("ITEM1" or "ITEMS1" or "ITEMI" or "ITEMSI" 
                                                             or "ITEML" or "ITEMSL")).any():
             start_index = (item_position["item_number"] == ("ITEM1" or "ITEMS1" or "ITEMI" or "ITEMSI" 
                                                             or "ITEML" or "ITEMSL")) 
        elif (item_position["item_number"] == ("PARTI" or "PARTL" or "PART1")).any():
             start_index = (item_position["item_number"] == ("PARTI" or "PARTL" or "PART1"))
        else:
            corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name, "section_error"] = 1
        #split the text such that it starts with Part 1 / item 1
        try: 
            html_text = html_text[item_position["start_position"][start_index].reset_index(drop=True)[0]:]    
        except (KeyError, TypeError):
            file_name = file_name.replace('.txt', '_SECTION_ERROR.txt')
            corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name, "section_error"] = 1

        
        if corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name.replace('_SECTION_ERROR.txt', '.txt'), "section_error"].item() == 0:
            print(f"Cleaning in process for filing year {i} and file: {file_name}")
        else:
            print(f"Section/Part 1 not found. Continue cleaning process with raw file for filing year {i} and file: {file_name.replace('_SECTION_ERROR.txt', '.txt')}")    

        #Encode the HTML string to bytes before passing it to fromstring function:
        ten_k_text = get_text_from_html(html_text.encode('utf-8'))
        
        #If further_clean_up is set to True, the text will be further cleaned up
        if further_clean_up == True:
            # Use re.sub to replace email adresses with the word "email" and websites with the word "website"
            ten_k_text = re.sub(r'(?:[a-z0-9!#$%&\'*+\/=?^_`{|}~-]+(?:\.[a-z0-9!#$%&\'*+\/=?^_`{|}~-]+)*|"(?:[\x01-\x08\x0b\x0c\x0e-\x1f\x21\x23-\x5b\x5d-\x7f]|\\[\x01-\x09\x0b\x0c\x0e-\x7f])*")\@(?:(?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.)+[a-z0-9](?:[a-z0-9-]*[a-z0-9])?|\[(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?|[a-z0-9-]*[a-z0-9]:(?:[\x01-\x08\x0b\x0c\x0e-\x1f\x21-\x5a\x53-\x7f]|\\[\x01-\x09\x0b\x0c\x0e-\x7f])+)\])', 'email-adress',ten_k_text).strip()

            ten_k_text = re.sub(r'\b((https|http):\/\/)*(www)*\.*\w+\.(com)\b', 'website',ten_k_text).strip()

            # Use re.sub to replace occurrences of line-breaks, tabulars, speical ASCII characters etc. with a single empty space
            # Remove tabulars
            ten_k_text = ten_k_text.replace('\t', ' ').strip() 
            # Remove specific ASCII characters and HTML entities
            ten_k_text = re.sub(r'[\x80-\xFF]', '', ten_k_text)  
            ten_k_text = re.sub(r'[\x91-\x97]', '', ten_k_text)
            ten_k_text = re.sub(r'&#\w{3};|\u200b', '',ten_k_text).strip() 

            # Replace remaining line breaks with a single empty space
            ten_k_text = re.sub(r'\n', ' ',ten_k_text).strip() 

            # Define characters which should be removed. Note ',.'´-!$%&()' will be removed at a later stage
            chars = '\"#*+•/:;<=>?@[\]^_—`{|}~™†®☒☐■☑●◦”“'
            ten_k_text = ten_k_text.translate(str.maketrans(chars, ' ' * len(chars))) 

            # Replace occurrences of multiple empty spaces with a single empty space
            ten_k_text = re.sub(r'\s{2,}', ' ', ten_k_text)

            # Replace the occurrence of multiple dots with a single dot
            ten_k_text = re.sub(r'\.+\s*\.*', '. ',ten_k_text).strip() 

        #Add the text to the temporary corpus data frame
        corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name.replace('_SECTION_ERROR.txt', '.txt'),"filing_text"] = ten_k_text[:]

        if save_single_file == True:
            #create the file name
            txt_name = file_name.replace('.txt', '_cleaned.txt')

            #create the path for the .txt file.
            txt_path = os.path.join(output_path,txt_name)

            # Save text to the specified path
            with open(txt_path, 'w', encoding='utf-8') as txt_file:
                txt_file.write(ten_k_text)
    
    #Add the temporary corpus data frame to the overall corpus data frame
    if corpus_df.empty == True:
         corpus_df = corpus_tmp_df
    else:
         corpus_df = pd.concat([corpus_df, corpus_tmp_df],ignore_index=True).reset_index(drop=True)

#save the whole data frame as pickle format
#with further clean up steps: Corpus_df_HTML_cleaned_v1
#without further clean up steps: Corpus_df_HTML_v1
corpus_df.to_pickle("BoW/Intermediate_datasets/Corpus_df_HTML_cleaned_BoW_v1.pkl")

#Note this is just an intermediate version of the corpus data frame. The final version will be saved after the BoW pre-processing steps are completed.

The second code is used for the GLLM and W2V approach.

In [ ]:
# Create new folder to save intermediate files
if not os.path.exists("GLLM/Intermediate_datasets"):
    os.makedirs("GLLM/Intermediate_datasets")

# Create a new dataframe to save the pre-processed files
corpus_df = pd.DataFrame()

#Step option to further clean up the text
further_clean_up = True

#Set option whether to save a single 10-k filing as a .txt file
save_single_file = False

#Loop through each year, pre-process the filings, and save them into the dataframe
for i in range(2013,2024):
    #Define the input-path (i.e. where you saved the 10-K filings to in Step 1)
    os_path = "10K/" + str(i)
    #Define the output-path
    output_path = "10K/" + str(i) + "/cleaned/"
    #Extract a list of all files included in the respective year-folder
    html_docs = os.listdir(os_path)
    
    if 'cleaned' in html_docs:
        html_docs.remove('cleaned')

    #Create a temporary dataframe for the respective filing-year
    corpus_tmp_df = pd.DataFrame()

    #Define first variables like "file_name", "filing_year", "CIK codes", and a empty column for the pre-processed filing text
    corpus_tmp_df["file_name"] = html_docs
    corpus_tmp_df["filing_year"] = i
    corpus_tmp_df["CIK"] = corpus_tmp_df["file_name"].str.slice(0,10)
    corpus_tmp_df["reporting_month"] = corpus_tmp_df["file_name"].str.slice(11,13)
    corpus_tmp_df["reporting_year"] = corpus_tmp_df["file_name"].str.slice(14,18)
    corpus_tmp_df["filing_text"] = None
    corpus_tmp_df["section_error"] = 0

    #loop through each filing for the respective filing year and pre-proccess it.
    for file_name in html_docs:

        file_path = os.path.join(os_path,file_name)
        with open(file_path, 'r', encoding='utf-8') as txt_file:
            html_text = txt_file.read()

        #get the position of the single items
        item_position = get_section_position(html_text)
        start_index = pd.NA

        #index for start items
        if item_position.empty:
            corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name, "section_error"] = 1
        elif (item_position["item_number"] == ("ITEM1" or "ITEMS1" or "ITEMI" or "ITEMSI" 
                                                             or "ITEML" or "ITEMSL")).any():
             start_index = (item_position["item_number"] == ("ITEM1" or "ITEMS1" or "ITEMI" or "ITEMSI" 
                                                             or "ITEML" or "ITEMSL")) 
        elif (item_position["item_number"] == ("PARTI" or "PARTL" or "PART1")).any():
             start_index = (item_position["item_number"] == ("PARTI" or "PARTL" or "PART1"))
        else:
            corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name, "section_error"] = 1
        #split the text such that it starts with Part 1 / item 1
        try: 
            html_text = html_text[item_position["start_position"][start_index].reset_index(drop=True)[0]:]    
        except (KeyError, TypeError):
            file_name = file_name.replace('.txt', '_SECTION_ERROR.txt')
            corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name, "section_error"] = 1

        
        if corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name.replace('_SECTION_ERROR.txt', '.txt'), "section_error"].item() == 0:
            print(f"Cleaning in process for filing year {i} and file: {file_name}")
        else:
            print(f"Section/Part 1 not found. Continue cleaning process with raw file for filing year {i} and file: {file_name.replace('_SECTION_ERROR.txt', '.txt')}")    

        #Encode the HTML string to bytes before passing it to fromstring function:
        ten_k_text = get_text_from_html(html_text.encode('utf-8'))
        
        #If further_clean_up is set to True, the text will be further cleaned up
        if further_clean_up == True:
            
            # Use re.sub to replace occurrences of line-breaks, tabulars, speical ASCII characters etc. with a single empty space
            # Remove tabulars
            ten_k_text = ten_k_text.replace('\t', ' ').strip() 
            # Remove specific ASCII characters and HTML entities
            ten_k_text = re.sub(r'[\x80-\xFF]', ' ', ten_k_text)  
            ten_k_text = re.sub(r'[\x91-\x97]', ' ', ten_k_text)
            ten_k_text = re.sub(r'&#\w{3};|\u200b', ' ',ten_k_text).strip() 

            # Replace line breaks directly followed by a word with a space
            ten_k_text = re.sub(r'\n(?=\w)', ' ',ten_k_text).strip() 

            # Replace multiple line breaks with just two
            ten_k_text = re.sub(r'(\n)+(\s*\n)+', '\n\n', ten_k_text).strip() 

            # Define characters which should be removed. Note ',.'´-!$%&()' will be removed at a later stage
            chars = '#*+•/<=>?[\]^_—`{|}~™†®☒☐■☑●◦'
            ten_k_text = ten_k_text.translate(str.maketrans(chars, ' ' * len(chars))) 

            # Replace the occurrence of multiple dots with a single dot
            ten_k_text = re.sub(r'\.+\s*\.*', '. ',ten_k_text).strip() 

        #Add the text to the temporary corpus data frame
        corpus_tmp_df.loc[corpus_tmp_df["file_name"] == file_name.replace('_SECTION_ERROR.txt', '.txt'),"filing_text"] = ten_k_text[:]

        if save_single_file == True:
            #create the file name
            txt_name = file_name.replace('.txt', '_cleaned.txt')

            #create the path for the .txt file.
            txt_path = os.path.join(output_path,txt_name)

            # Save text to the specified path
            with open(txt_path, 'w', encoding='utf-8') as txt_file:
                txt_file.write(ten_k_text)
    
    #Add the temporary corpus data frame to the overall corpus data frame
    if corpus_df.empty == True:
         corpus_df = corpus_tmp_df
    else:
         corpus_df = pd.concat([corpus_df, corpus_tmp_df],ignore_index=True).reset_index(drop=True)

#save the whole data frame as pickle format
corpus_df.to_pickle("GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_v1.pkl")

Next, we will add some furhter information to the corpus dataframe. This includes the company name, the accession number, and a word count variable.

In [ ]:
# Load the dataframe from the pickle file created before
#with further clean up steps BoW: Corpus_df_HTML_cleaned_BoW_v1
#with further clean up steps W2V and GLLM: Corpus_df_HTML_cleaned_GLLM_v1

file_paths = ["BoW/Intermediate_datasets/Corpus_df_HTML_cleaned_BoW_v1.pkl","GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_v1.pkl"]

for file_path in file_paths:
    with open(file_path, 'rb') as file:
        corpus_df = pickle.load(file)

    #Load the additional filings info
    with open('S_and_P_500_filings_info.pkl', 'rb') as file:
        S_and_P_500_filings = pickle.load(file)

    #Create unique cik_year keys to match furhter information
    corpus_df["filing_key"] = corpus_df["CIK"].astype(str) + "_" + corpus_df["filing_year"].astype(str)

    #match company name and accession number to the dataframe
    corpus_df = pd.merge(corpus_df,S_and_P_500_filings[["company","accession_number","filing_key"]],how='left',on='filing_key')

    #Get the total number of words in each filing
    corpus_df["Word_count"] = corpus_df["filing_text"].apply(lambda x: len(re.findall(r'\b[a-zA-Z]+\b',str(x))))

    #save the whole data frame as pickle format

    corpus_df.to_pickle(file_path)